# Regressão linear 

In [60]:
import pandas as pd
import statsmodels.formula.api as smf

Leitura do dataset.

In [61]:
caminho = r'C:\Users\saulosgil\OneDrive\Documentos\health_outcomes_workers\data\df_analise.csv'
df = pd.read_csv(caminho)

df.head()

,idade,sexo,estado_civil,escolaridade,renda_mensal,cor_raca,tempo_trab_anos,funcao_trab,carga_horaria_horas_sem,outro_trab,...,Diuretics,Beta_blockers,ACE_inhibitors,Metformin,CCB,whoqol_fisico,whoqol_psicologico,whoqol_social,whoqol_ambiente,whoqol_n_validos
0,34,feminino,separado,médio completo,1 a 3 salários mínimos,preto (a),1.0,cooperado,40,sim,...,0,0,0,0,0,71.43,70.83,83.33,78.12,26
1,59,masculino,divorciado,médio completo,mais de 3 salários,pardo (a),2.0,cooperado,56,não,...,0,0,0,0,0,100.00,83.33,100.00,93.75,26
2,27,masculino,solteiro,superior completo,mais de 3 salários,pardo (a),1.0,cooperado,44,não,...,0,0,0,0,0,78.57,58.33,75.00,75.00,26
3,40,masculino,solteiro,médio completo,1 a 3 salários mínimos,preto (a),1.0,cooperado,45,não,...,0,0,0,0,0,78.57,95.83,100.00,78.12,26
4,23,masculino,solteiro,médio completo,mais de 3 salários,preto (a),2.0,cooperado,45,não,...,0,0,0,0,0,75.00,83.33,66.67,90.62,26


Ajuste das variáveis para aplicar no modelo.

In [62]:
df['funcao_trab'] = df['funcao_trab'].replace({
    'cooperado':   'Cooperative waste pickers',
    'individual':  'Independent waste pickers',
    'Cooperative': 'Cooperative waste pickers',
    'Independent': 'Independent waste pickers',
})

# Binária: < 65 = 1, >= 65 = 0, ausente = NaN
df['idade_cat'] = np.where(df['idade'] < 65, '< 65 anos', '>= 65 anos')

# Regressão linear para todas as variáveis

1. Desfechos
    - `peso`;                 
    - `imc`;                  
    - `circun_abdome`;        
    - `cintura_altura_ratio`; 
    - `whoqol_fisico`;        
    - `whoqol_psicologico`;   
    - `whoqol_social`;        
    - `whoqol_ambiente`.      

2. Covariáveis
    - `sexo`;                 
    - `idade_cat`;                  
    - `cor_raca`;        
    - `renda_mensal`; 
    - `DM2`;        
    - `HAS`.

In [63]:

ref   = "C(funcao_trab, Treatment('Cooperative waste pickers'))"
termo = f"{ref}[T.Independent waste pickers]"

desfechos = {
    'peso':                 'Weight (kg)',
    'imc':                  'BMI (kg/m²)',
    'circun_abdome':        'Waist circumference (cm)',
    'cintura_altura_ratio': 'Waist-to-height ratio (×100)',
    'whoqol_fisico':        'whoqol_fisicoo',
    'whoqol_psicologico':   'Wwhoqol_psicologico',
    'whoqol_social':        'whoqol_social',
    'whoqol_ambiente':      'whoqol_ambiente',
}

# Conjunto mínimo de ajuste (confundidores plausíveis)

covariaveis = ['C(sexo)', 'C(idade_cat)', 'C(cor_raca)', 'C(renda_mensal)', 'C(DM2)', 'C(HAS)']

resultados_reg = []

for var, rotulo in desfechos.items():
    formula = f"{var} ~ {ref} + {' + '.join(covariaveis)}"
    m = smf.ols(formula, data=df).fit(cov_type='HC3')   # EP robustos, adequados para N pequeno

    b      = m.params[termo]
    lo, hi = m.conf_int().loc[termo]
    p      = m.pvalues[termo]
    n_par  = len(m.params)


    resultados_reg.append({
        'Outcome': rotulo,
        'β (IC 95%)': f'{b:.2f} ({lo:.2f} a {hi:.2f})',
        'P': round(p, 4),
        'n': int(m.nobs),
        'R² ajustado': round(m.rsquared_adj, 2),
    })

tabela_reg = pd.DataFrame(resultados_reg)
tabela_reg

,Outcome,β (IC 95%),P,n,R² ajustado
0,Weight (kg),-16.77 (-27.39 a -6.15),0.0020,44,0.19
1,BMI (kg/m²),-5.05 (-8.82 a -1.28),0.0086,44,0.21
2,Waist circumference (cm),-13.69 (-22.49 a -4.88),0.0023,44,0.18
3,Waist-to-height ratio (×100),-7.12 (-12.65 a -1.60),0.0115,44,0.24
4,whoqol_fisicoo,1.46 (-14.20 a 17.13),0.8546,44,0.11
5,Wwhoqol_psicologico,-3.98 (-18.22 a 10.25),0.5834,44,0.08
6,whoqol_social,-27.46 (-56.69 a 1.76),0.0655,44,0.09
7,whoqol_ambiente,-11.74 (-25.70 a 2.21),0.0991,44,0.06
